# Soluciones · 02 Errores sistemáticos

Soluciones resueltas de los ejercicios del [cuaderno 02](../../notebooks/02_errores_sistematicos.ipynb). Intenta antes cada ejercicio por tu cuenta.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/telmomm/nanovna-measurement-course

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from vnacourse import plotting
from vnacourse.data import load_repetitions
from vnacourse.errormodel import ErrorTerms, apply_error_model, synthetic_error_terms
from vnacourse.reflection import z_to_gamma

plotting.use_course_style()

f_hz = np.linspace(50e3, 1.45e9, 401)
f_mhz = f_hz / 1e6
terms = synthetic_error_terms(f_hz)

## 1. ¿Qué término tiene la culpa?

Construye cuatro instrumentos: el completo y tres con un término hecho perfecto. `ErrorTerms` es una tupla con nombre, así que `_replace` da una copia con un campo cambiado. El tamaño del rizado es la diferencia entre la lectura más alta y la más baja.

In [ ]:
zeros = np.zeros(f_hz.shape, dtype=complex)
instruments = {
    "los tres términos": terms,
    "sin directividad": terms._replace(directivity=zeros),
    "sin desadaptación": terms._replace(source_match=zeros),
    "seguimiento perfecto": terms._replace(tracking=zeros + 1),
}

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharex=True)
for ax, (device, gamma) in zip(
    axes, (("resistencia de 75 Ω", z_to_gamma(75)), ("cortocircuito", -1.0)), strict=True
):
    print(device)
    for name, instrument in instruments.items():
        measured = np.abs(apply_error_model(gamma + zeros, instrument))
        ax.plot(f_mhz, measured, label=name)
        print(f"   {name:<20} rizado {measured.max() - measured.min():.3f}")
    ax.set(xlabel="frecuencia (MHz)", ylabel="$|\\Gamma_m|$")
    ax.set_title(device)
axes[0].legend(loc="upper center", bbox_to_anchor=(1.1, -0.18), ncol=4);

- **Resistencia de 75 Ω:** eliminar la directividad hace que el rizado caiga de 0.18 a 0.04. Eliminar la desadaptación de fuente casi no cambia nada. Para una reflexión pequeña, la **directividad** es el término que importa: es un vector de tamaño 0.08 sumado a una reflexión de tamaño 0.2.
- **Cortocircuito:** ahora eliminar la **desadaptación de fuente** es lo que más efecto tiene: reduce el rizado a la mitad. La directividad sigue contribuyendo, pero 0.08 es poco al lado de una reflexión de tamaño 1, mientras que el término de desadaptación $e_{11}\Gamma$ está en su máximo.
- Hacer perfecto el **seguimiento** cambia más la *forma* del rizado que su tamaño. El retardo del término de seguimiento es lo que hace que la reflexión gire deprisa respecto a la directividad; sin él, los dos vectores siguen sumándose y cancelándose, solo que más despacio con la frecuencia.

La respuesta a «qué error importa más» depende del dispositivo. Es otra vez la tabla de la sección 2.3, vista a lo largo de la frecuencia.

## 2. Rizado y longitud

Un instrumento cuyo único defecto es una desadaptación de fuente $0.2\,e^{-j2\pi f\tau}$, midiendo un cortocircuito. Barremos una banda más ancha para que quepan varios rizados, y buscamos los picos como los puntos más altos que sus dos vecinos.

In [ ]:
f_wide = np.linspace(1e6, 6e9, 6001)

fig, ax = plt.subplots()
for tau in (0.5e-9, 1e-9, 2e-9):
    instrument = ErrorTerms(
        directivity=0, source_match=0.2 * np.exp(-2j * np.pi * f_wide * tau), tracking=1
    )
    measured = np.abs(apply_error_model(-1.0, instrument))
    ax.plot(f_wide / 1e9, measured, label=f"τ = {tau * 1e9:g} ns")

    peaks = np.flatnonzero((measured[1:-1] > measured[:-2]) & (measured[1:-1] > measured[2:])) + 1
    spacing = np.diff(f_wide[peaks]).mean()
    print(
        f"τ = {tau * 1e9:g} ns: picos separados {spacing / 1e6:.0f} MHz;  "
        f"1/τ = {1 / tau / 1e6:.0f} MHz"
    )
ax.set(xlabel="frecuencia (GHz)", ylabel="$|\\Gamma_m|$ de un cortocircuito")
ax.set_title("Rizado para tres retardos de la desadaptación de fuente")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3);

Los picos están separados $1/\tau$: el rizado se repite cada vez que la fase de $e_{11}$ completa una vuelta. La profundidad del rizado es la misma en los tres casos (depende de $|e_{11}|$); solo cambia su separación.

En un instrumento real, $\tau$ es el retardo entre el DUT y el lugar donde se produce la segunda reflexión. Un cable largo significa un $\tau$ grande y, por tanto, un **rizado fino y apretado**. Puedes darle la vuelta: mide la separación de un rizado y sabrás a qué distancia está la desadaptación que lo produce. Con $\tau$ como retardo de ida y vuelta en un cable a $0.7c$, un rizado cada 100 MHz apunta a una desadaptación a aproximadamente 1 m.

## 3. El peor caso

La lectura es la suma de dos vectores de longitud 0.1 (el dispositivo) y 0.03 (la directividad), con fase relativa desconocida. Es máxima cuando están alineados y mínima cuando se oponen.

In [ ]:
gamma_true, directivity = 0.1, 0.03
for label, reading in (
    ("lectura mínima", gamma_true - directivity),
    ("valor real", gamma_true),
    ("lectura máxima", gamma_true + directivity),
):
    print(
        f"{label:<15} |Γ| = {reading:.2f}   pérdidas de retorno = {-20 * np.log10(reading):.1f} dB"
    )

Un dispositivo con 20 dB de pérdidas de retorno puede leer cualquier valor entre 17.7 dB y 23.1 dB. El error no es simétrico en decibelios y crece deprisa a medida que el dispositivo es mejor: para unas pérdidas de retorno reales de 30 dB ($|\Gamma| = 0.032$), una directividad de 0.03 hace que la lectura no signifique nada. La directividad de un instrumento, tras la calibración, fija la mejor adaptación que puede certificar.

## 4. Datos reales

In [ ]:
f_real, open_raw = load_repetitions("standards", "open")
_, short_raw = load_repetitions("standards", "short")
f_real_mhz = f_real / 1e6

fig, (ax_phase, ax_reps) = plt.subplots(1, 2, figsize=(10, 4.2))
ax_phase.plot(f_real_mhz, np.degrees(np.angle(open_raw.mean(axis=0))), label="abierto (ideal: 0°)")
ax_phase.plot(
    f_real_mhz,
    np.degrees(np.angle(-short_raw.mean(axis=0))),
    label="cortocircuito, menos 180° (ideal: 0°)",
)
ax_phase.set(xlabel="frecuencia (MHz)", ylabel="error de fase (°)")
ax_phase.set_title("Fase de los estándares en bruto")
ax_phase.legend(loc="lower right")

for repetition in open_raw:
    ax_reps.plot(f_real_mhz, np.abs(repetition), color=plotting.SERIES[0], linewidth=0.6)
ax_reps.set(xlabel="frecuencia (MHz)", ylabel="$|\\Gamma_m|$")
ax_reps.set_title("Abierto en bruto: las diez repeticiones")

above = f_real > 150e6
phase_open = np.degrees(np.angle(open_raw.mean(axis=0)))
print(
    f"Fase del abierto: de {phase_open.min():.0f}° a {phase_open.max():.0f}° en toda la banda, "
    f"dentro de ±{np.abs(phase_open[above]).max():.1f}° por encima de 150 MHz"
)

difference = np.abs(np.abs(open_raw) - np.abs(open_raw.mean(axis=0)))
print(
    f"Las repeticiones difieren de su media {np.median(difference):.4f} (típico), "
    f"{difference.max():.2f} (peor caso)"
)

La fase cuenta la misma historia que el módulo. Por debajo de unos 150 MHz se desvía hasta 100°; por encima, se mantiene a menos de 1° del valor ideal. Un error de fase es el argumento del término de seguimiento (sección 2.2).

Las diez repeticiones están unas encima de otras: a esta escala no se distinguen, y su diferencia típica está por debajo de 0.001. Esa es la gracia del ejercicio. **El error grande es perfectamente repetible**, así que es sistemático, y repetir la medida no sirve para revelarlo. Los pocos picos visibles son otro asunto, que se estudia en el cuaderno 05.